# Вступ до PySpark (огляд на високому рівні)

**Apache Spark** — фреймворк для розподіленої обробки великих даних.
**PySpark** — Python API для Spark.

---
**Рівень:** Junior Data Engineer  
**Мета:** зрозуміти, коли і чому Spark, а не Pandas

In [ ]:
try:
    from pyspark.sql import SparkSession
    import pyspark
    print(f'PySpark version: {pyspark.__version__}')
    SPARK_AVAILABLE = True
except ImportError:
    print('PySpark не встановлено. Встановіть: pip install pyspark')
    print('Код буде показаний для ознайомлення.')
    SPARK_AVAILABLE = False

---
## 1. Що таке Apache Spark?

Apache Spark — **двигун розподіленої обробки даних** (distributed computing engine).

### Ключові ідеї:
- Обробляє дані **в пам'яті** (in-memory), а не на диску (як Hadoop MapReduce)
- Автоматично **розподіляє** дані та обчислення по кластеру
- Працює з даними, які не влазять в RAM однієї машини (терабайти+)
- Підтримує **lazy evaluation** — обчислення тільки коли потрібен результат

### Компоненти Spark:

```
       Spark SQL         Structured Streaming     MLlib     GraphX
       (DataFrames/SQL)   (streaming data)         (ML)      (graphs)
                               |
                     Spark Core (RDD API)
                               |
              Cluster Manager (Standalone / YARN / K8s / Mesos)
```

---
## 2. Spark vs Pandas

Головне питання: **коли Pandas, а коли Spark?**

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    ['Обсяг даних', '< RAM (1-10 ГБ)', '>> RAM (ТБ-ПБ)'],
    ['Обробка', 'Одне ядро (1 CPU)', 'Розподілено (N CPU/Nodes)'],
    ['Оцінка (Evaluation)', 'Eager (одразу)', 'Lazy (при дії)'],
    ['Індекс', 'Є (важлива частина)', 'Немає (тільки дані)'],
    ['SQL', 'Через read_sql', 'Нативний Spark SQL'],
    ['Коли використовувати', 'Прототипи, малі дані, ML', 'Продакшн, великі дані, ETL'],
    ['Складність', 'Низька', 'Середня (кластер, конфігурація)'],
    ['Вартість', '1 машина', 'Кластер (N машин)'],
], columns=['Критерій', 'Pandas', 'PySpark'])

display(comparison)

---
## 3. Основні концепції Spark

### 3.1 RDD (Resilient Distributed Dataset)
- Низькорівневе API (не рекомендується для початківців)
- Розподілений набір даних, стійкий до відмов

### 3.2 DataFrame 🎯
- Високорівневе API (як Pandas, але розподілено)
- Оптимізований через **Catalyst Optimizer**
- Це те, з чим ви будете працювати 90% часу

### 3.3 Lazy Evaluation
- **Transformation** — описуємо що хочемо (фільтр, groupby, join)
- **Action** — запускаємо обчислення (.show(), .count(), .write())
- Spark будує **DAG** (Directed Acyclic Graph) і оптимізує його

```
df = spark.read.csv(...)       # Transformation (lazy)
df_filtered = df.filter(...)   # Transformation (lazy)
df_grouped = df_filtered.groupBy(...).count()  # Transformation (lazy)
df_grouped.show()              # Action! Тільки тепер Spark виконує
```

In [ ]:
# 3.4 SparkSession — вхідна точка
if SPARK_AVAILABLE:
    spark = SparkSession.builder \
        .appName('PySpark_Intro') \
        .master('local[*]') \
        .config('spark.sql.repl.eagerEval.enabled', True) \
        .getOrCreate()
    
    print('SparkSession створено')
    print(f'Spark UI: {spark.sparkContext.uiWebUrl}')
    print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')
else:
    print('Для запуску: pip install pyspark')
    print('''
    spark = SparkSession.builder \\
        .appName('PySpark_Intro') \\
        .master('local[*]') \\
        .getOrCreate()
    ''')

---
## 4. DataFrame API — перше знайомство

Синтаксис дуже схожий на Pandas, але є ключові відмінності.

In [ ]:
if SPARK_AVAILABLE:
    # Створення DataFrame вручну
    data = [
        ('Alice', 'IT', 70000, 25),
        ('Bob', 'HR', 50000, 30),
        ('Charlie', 'IT', 80000, 35),
        ('Diana', 'Finance', 55000, 28),
        ('Eve', 'HR', 65000, 40)
    ]
    columns = ['name', 'department', 'salary', 'age']

    df = spark.createDataFrame(data, schema=columns)
    
    print('=== DataFrame ===')
    df.show()
    
    print('\n=== Schema ===')
    df.printSchema()
    
    print('\n=== Count ===')
    print(f'Rows: {df.count()}')

In [ ]:
# Pandas-style код (для порівняння) — якби те саме в Pandas
df_pd = pd.DataFrame(data, columns=columns)
print('Pandas version:')
display(df_pd)

In [ ]:
# Основні операції в PySpark
if SPARK_AVAILABLE:
    from pyspark.sql.functions import col, avg, count, when, isnan
    
    # select (вибір колонок)
    df.select('name', 'salary').show(5)
    
    # filter (where)
    df.filter(col('salary') > 55000).show()
    
    # groupBy
    df.groupBy('department').agg(
        avg('salary').alias('avg_salary'),
        count('*').alias('emp_count')
    ).show()
    
    # withColumn (нова колонка)
    df.withColumn('salary_category', 
        when(col('salary') > 70000, 'high')
        .when(col('salary') > 55000, 'medium')
        .otherwise('low')
    ).show()
    
    # sort
    df.sort(col('salary').desc()).show(3)

---
## 5. Читання та запис даних

Spark підтримує всі популярні формати.

In [ ]:
# Шаблони читання (запуститься якщо є Spark)
if SPARK_AVAILABLE:
    print('Приклади читання (закоментировано — файли не створені):')
    print('''
    # CSV
    df_csv = spark.read.csv('path/file.csv', header=True, inferSchema=True)
    
    # JSON
    df_json = spark.read.json('path/file.json')
    
    # Parquet (оптимальний формат для Spark!)
    df_pq = spark.read.parquet('path/file.parquet')
    
    # З таблиці БД (через JDBC)
    df_jdbc = spark.read \\
        .format('jdbc') \\
        .option('url', 'jdbc:postgresql://host/db') \\
        .option('dbtable', 'schema.table') \\
        .option('user', 'user') \\
        .option('password', 'pass') \\
        .load()
    ''')
    
    print('Запис:')
    print('''
    df.write.mode('overwrite').parquet('output/result.parquet')
    df.write.mode('append').csv('output/result.csv')
    ''')

---
## 6. Spark SQL

Можна писати SQL-запити напряму до DataFrame.

In [ ]:
if SPARK_AVAILABLE:
    # Реєструємо DataFrame як тимчасову таблицю
    df.createOrReplaceTempView('employees')
    
    # SQL-запит
    result = spark.sql('''
        SELECT 
            department,
            COUNT(*) AS cnt,
            ROUND(AVG(salary), 0) AS avg_salary,
            ROUND(AVG(age), 1) AS avg_age
        FROM employees
        WHERE salary > 50000
        GROUP BY department
        ORDER BY avg_salary DESC
    ''')
    
    print('=== Spark SQL результат ===')
    result.show()
else:
    print('''
    df.createOrReplaceTempView('employees')
    result = spark.sql('SELECT department, COUNT(*) FROM employees GROUP BY department')
    result.show()
    ''')

---
## 7. Коли використовувати Spark в DE?

### ✅ Використовуйте Spark коли:
- **Дані > 100 ГБ** — не влазять в Pandas
- **Розподілені обчислення** — потрібна кластеризація
- **ETL на великих даних** — щоденна обробка терабайтів
- **Data Lake** — Parquet, Delta Lake, Iceberg
- **Streaming** — реальний час (Kafka + Spark Streaming)

### ❌ Не використовуйте Spark коли:
- Дані < 10 ГБ (Pandas швидше і простіше)
- Потрібна візуалізація (matplotlib, seaborn)
- Прототипування / ad-hoc аналіз
- Немає кластера / ресурсів

In [ ]:
# Decision Tree: Pandas vs Spark
print('''
╔═══════════════════════════════════════════════╗
║           Pandas vs Spark Decision           ║
╠═══════════════════════════════════════════════╣
║  Дані влазять в RAM?                         ║
║  ├─ Так → Дані < 10 ГБ?                      ║
║  │   ├─ Так → Pandas ✅                       ║
║  │   └─ Ні → Spark обережно (local mode)     ║
║  └─ Ні → Spark ✅ (кластер)                  ║
║                                               ║
║  Потрібен Streaming?                          ║
║  ├─ Так → Spark Structured Streaming ✅       ║
║  └─ Ні → Залежить від об'єму                 ║
╚═══════════════════════════════════════════════╝
''')

---
## 8. Обмеження та особливості PySpark

| Проблема | Опис | Рішення |
|----------|------|---------|
| **Overhead** | Spark запускає JVM, навіть для 1 МБ даних | Використовуйте Pandas для малих даних |
| **Латентність** | ~5-10с на старт SparkSession | Тримайте сесію відкритою |
| **Пам'ять** | OutOfMemory при неправильній конфігурації | Налаштуйте spark.executor.memory |
| **UDF** | Python UDF повільні (серіалізація) | Використовуйте вбудовані функції |
| **Shuffle** | join/groupBy може створити гігантський shuffle | Оптимізуйте bucketing |
| **Шумні сусіди** | Спільний кластер — ресурси нестабільні | Використовуйте Delta Lake / Lakehouse |

> **Головне правило DE**: починайте з Pandas, переходьте на Spark коли Pandas 'впирається' в пам'ять або час.

In [ ]:
# Зупинка SparkSession (якщо створювали)
if SPARK_AVAILABLE and 'spark' in dir():
    spark.stop()
    print('SparkSession зупинено')

---
## Резюме

| Тема | Суть |
|------|------|
| **Apache Spark** | Розподілений двигун обробки даних (ТБ-ПБ) |
| **PySpark** | Python API для Spark |
| **DataFrame** | Основне API (схоже на Pandas) |
| **Lazy Evaluation** | Описуємо → виконуємо при action |
| **Spark SQL** | SQL-запити до DataFrame |
| **Коли використовувати** | Дані не влазять в Pandas (>100ГБ) |
| **Parquet** | Основний формат для Spark (columnar, стиснення, schema) |

> **Далі:** `lab2/` (Pandas + БД) і `lab3/` (Pandas + API) — закріпіть Pandas, потім переходьте до Spark.

In [ ]:
# Корисні посилання
print(''''
🔗 Корисні посилання:
- Apache Spark офіційний сайт: https://spark.apache.org/
- PySpark Documentation: https://spark.apache.org/docs/latest/api/python/
- Databricks PySpark Guide: https://www.databricks.com/pyspark-guide
- Spark in Action (книга): https://www.manning.com/books/spark-in-action
- Встановлення: pip install pyspark
''')